In [53]:
# !pip install xgboost

### Import all the libraries

In [54]:
#  Core Libraries 
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#  Dataset & Preprocessing 
from sklearn.datasets import load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    StratifiedKFold, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn import metrics 
#  Classifiers 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier

#  Distributions for RandomizedSearch 
from scipy.stats import randint, uniform, loguniform

### Load the dataset

In [55]:
iris = load_iris()
iris

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [56]:
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

In [57]:
print('Shape       :', X.shape)
print('Classes     :', list(iris.target_names))
print('Missing vals:', X.isnull().sum().sum())

Shape       : (150, 4)
Classes     : [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Missing vals: 0


In [58]:
X.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


### Train-Test Split

In [59]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((120, 4), (30, 4), (120,), (30,))

In [60]:
np.unique(y)

array([0, 1, 2])

In [61]:
pd.Series(y).value_counts()

0    50
1    50
2    50
Name: count, dtype: int64

### Scale Features (required for LR, KNN, SVM)

In [62]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [63]:
def print_metrics(model_name, y_test, y_pred, y_pred_train, y_train):
    """Prints all evaluation metrics for a trained classifier."""
    print(f"\n{'='*55}")
    print(f"  MODEL : {model_name}")
    print(f"{'='*55}")

    #  Core Metrics 
    print(f"  Accuracy  (test)   : {metrics.accuracy_score(y_test, y_pred):.4f}")
    print(f"  Accuracy  (train)  : {metrics.accuracy_score(y_train, y_pred_train):.4f}")
    print(f"  Precision : {metrics.precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"  Recall    : {metrics.recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"  F1 Score  : {metrics.f1_score(y_test, y_pred, average='macro'):.4f}")
 
    #  Classification Report 
    print(f"\n  --- Classification Report ---")
    print(metrics.classification_report(y_test, y_pred, zero_division=0))

    #  Confusion Matrix 
    print(f"  --- Confusion Matrix ---")
    print(metrics.confusion_matrix(y_test, y_pred))


- 'macro' → Treat all classes equally (good for balanced evaluation)
- 'weighted' → Accounts for class imbalance (most commonly used)
- 'micro' → Based on total counts (useful for overall performance)

### Logistic Regression

In [64]:
from sklearn.linear_model import LogisticRegression
log_Class = LogisticRegression()
log_Class.fit(X_train_scaled,y_train)
y_pred_log = log_Class.predict(X_test_scaled)
y_predProb = log_Class.predict_proba(X_test_scaled) 
y_predtrain_log = log_Class.predict(X_train_scaled)


In [65]:
print_metrics("Logistic Regression", y_test, y_pred_log, y_predtrain_log, y_train)



  MODEL : Logistic Regression
  Accuracy  (test)   : 0.9333
  Accuracy  (train)  : 0.9583
  Precision : 0.9333
  Recall    : 0.9333
  F1 Score  : 0.9333

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


In [66]:
# ROC-AUC (needs predict_proba)
y_prob_log = log_Class.predict_proba(X_test_scaled)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_log, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_log):.4f}")

  ROC-AUC  (macro): 0.9967
  Log Loss           : 0.1740


- 'ovr' (One-vs-Rest)
    - Class 0 vs (1,2)
    - Class 1 vs (0,2)
    - Class 2 vs (0,1)
- 'ovo' (One-vs-One)
    - 0 vs 1
    - 0 vs 2
    - 1 vs 2

### KNN Classifier

In [67]:
from sklearn.neighbors import KNeighborsClassifier

knn_Class = KNeighborsClassifier()
knn_Class.fit(X_train_scaled, y_train)

y_pred_knn       = knn_Class.predict(X_test_scaled)
y_predtrain_knn  = knn_Class.predict(X_train_scaled)
knn_Class.predict_proba(X_test_scaled)

print_metrics("KNN", y_test, y_pred_knn, y_predtrain_knn, y_train)

y_prob_knn = knn_Class.predict_proba(X_test_scaled)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_knn, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_knn):.4f}")


  MODEL : KNN
  Accuracy  (test)   : 0.9333
  Accuracy  (train)  : 0.9750
  Precision : 0.9444
  Recall    : 0.9333
  F1 Score  : 0.9327

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.83      1.00      0.91        10
           2       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]
  ROC-AUC  (macro): 0.9933
  Log Loss           : 0.1196


### SVM Classifier


In [68]:
from sklearn.svm import SVC

svm_Class = SVC(probability=True)   # probability=True enables predict_proba
svm_Class.fit(X_train_scaled, y_train)

y_pred_svm       = svm_Class.predict(X_test_scaled)
y_predtrain_svm  = svm_Class.predict(X_train_scaled)
svm_Class.predict_proba(X_test_scaled)

print_metrics("SVM", y_test, y_pred_svm, y_predtrain_svm, y_train)

y_prob_svm = svm_Class.predict_proba(X_test_scaled)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_svm, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_svm):.4f}")


  MODEL : SVM
  Accuracy  (test)   : 0.9667
  Accuracy  (train)  : 0.9750
  Precision : 0.9697
  Recall    : 0.9667
  F1 Score  : 0.9666

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.90      0.95        10
           2       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
  ROC-AUC  (macro): 0.9967
  Log Loss           : 0.1073


### Decision Tree Classifier

In [69]:
from sklearn.tree import DecisionTreeClassifier

dt_Class = DecisionTreeClassifier(random_state=42)
dt_Class.fit(X_train, y_train)   # tree-based: no scaling needed

y_pred_dt       = dt_Class.predict(X_test)
y_predtrain_dt  = dt_Class.predict(X_train)
dt_Class.predict_proba(X_test)

print_metrics("Decision Tree", y_test, y_pred_dt, y_predtrain_dt, y_train)

y_prob_dt = dt_Class.predict_proba(X_test)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_dt, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_dt):.4f}")


  MODEL : Decision Tree
  Accuracy  (test)   : 0.9333
  Accuracy  (train)  : 1.0000
  Precision : 0.9333
  Recall    : 0.9333
  F1 Score  : 0.9333

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
  ROC-AUC  (macro): 0.9500
  Log Loss           : 2.4029


### Random Forest Classifier

In [70]:
from sklearn.ensemble import RandomForestClassifier

rf_Class = RandomForestClassifier(random_state=42)
rf_Class.fit(X_train, y_train)

y_pred_rf       = rf_Class.predict(X_test)
y_predtrain_rf  = rf_Class.predict(X_train)
rf_Class.predict_proba(X_test)

print_metrics("Random Forest", y_test, y_pred_rf, y_predtrain_rf, y_train)

y_prob_rf = rf_Class.predict_proba(X_test)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_rf, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_rf):.4f}")


  MODEL : Random Forest
  Accuracy  (test)   : 0.9000
  Accuracy  (train)  : 1.0000
  Precision : 0.9024
  Recall    : 0.9000
  F1 Score  : 0.8997

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.82      0.90      0.86        10
           2       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0  9  1]
 [ 0  2  8]]
  ROC-AUC  (macro): 0.9867
  Log Loss           : 0.2263


### ADA Boost Classifier

In [71]:
from sklearn.ensemble import AdaBoostClassifier

ada_Class = AdaBoostClassifier(random_state=42)
ada_Class.fit(X_train, y_train)

y_pred_ada       = ada_Class.predict(X_test)
y_predtrain_ada  = ada_Class.predict(X_train)
ada_Class.predict_proba(X_test)

print_metrics("AdaBoost", y_test, y_pred_ada, y_predtrain_ada, y_train)

y_prob_ada = ada_Class.predict_proba(X_test)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_ada, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_ada):.4f}")


  MODEL : AdaBoost
  Accuracy  (test)   : 0.9333
  Accuracy  (train)  : 1.0000
  Precision : 0.9333
  Recall    : 0.9333
  F1 Score  : 0.9333

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
  ROC-AUC  (macro): 0.9750
  Log Loss           : 0.9391


### Gradient Boosting Classifier

In [72]:
from sklearn.ensemble import GradientBoostingClassifier

gb_Class = GradientBoostingClassifier(random_state=42)
gb_Class.fit(X_train, y_train)

y_pred_gb       = gb_Class.predict(X_test)
y_predtrain_gb  = gb_Class.predict(X_train)
gb_Class.predict_proba(X_test)

print_metrics("Gradient Boosting", y_test, y_pred_gb, y_predtrain_gb, y_train)

y_prob_gb = gb_Class.predict_proba(X_test)
print(f"  ROC-AUC  (macro): {metrics.roc_auc_score(y_test, y_prob_gb, multi_class='ovr', average='macro'):.4f}")
print(f"  Log Loss           : {metrics.log_loss(y_test, y_prob_gb):.4f}")


  MODEL : Gradient Boosting
  Accuracy  (test)   : 0.9667
  Accuracy  (train)  : 1.0000
  Precision : 0.9697
  Recall    : 0.9667
  F1 Score  : 0.9666

  --- Classification Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      0.90      0.95        10
           2       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

  --- Confusion Matrix ---
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
  ROC-AUC  (macro): 0.9900
  Log Loss           : 0.3629


In [73]:
# ── Define All Models with Default Parameters ──
models = {
    'Logistic Regression' : LogisticRegression(random_state=42),
    'KNN'                 : KNeighborsClassifier(),
    'SVM'                 : SVC(random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(random_state=42),
    'AdaBoost'            : AdaBoostClassifier(random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(random_state=42),
    'XGBoost'             : XGBClassifier(random_state=42,
                                          use_label_encoder=False,
                                          eval_metric='mlogloss'),
}

# Models that need scaled data
scaled_models = {'Logistic Regression', 'KNN', 'SVM'}

# ─ Train &  ──
results = {}
print(f"{'Model':<24} {'Train Acc':>10} {'Test Acc':>10}")
print('-' * 46)

for name, model in models.items():
    Xtr = X_train_sc if name in scaled_models else X_train
    Xte = X_test_sc  if name in scaled_models else X_test

    model.fit(Xtr, y_train)
    train_acc = accuracy_score(y_train, model.predict(Xtr))
    test_acc  = accuracy_score(y_test,  model.predict(Xte))
    results[name] = test_acc
    print(f'{name:<24} {train_acc:>10.4f} {test_acc:>10.4f}')

print(f'\nBest Model: {max(results, key=results.get)} ({max(results.values()):.4f})')

Model                     Train Acc   Test Acc
----------------------------------------------
Logistic Regression          0.9583     0.9333
KNN                          0.9750     0.9333
SVM                          0.9750     0.9667
Decision Tree                1.0000     0.9333
Random Forest                1.0000     0.9000
AdaBoost                     1.0000     0.9333
Gradient Boosting            1.0000     0.9667
XGBoost                      1.0000     0.9333

Best Model: SVM (0.9667)



##   Parameters Guide (All 8 Algorithms)

### 1. Logistic Regression
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `C` | Inverse regularization strength | 0.001, 0.1, 1, 10, 100 |
| `penalty` | Regularization type | 'l1', 'l2', 'elasticnet' |
| `solver` | Optimization algorithm | 'lbfgs', 'liblinear', 'saga' |
| `max_iter` | Max iterations | 100, 500, 1000 |
| `class_weight` | Handle imbalance | None, 'balanced' |

### 2. K-Nearest Neighbors
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `n_neighbors` | Number of neighbors (k) | 3, 5, 7, 9, 11 |
| `weights` | Voting strategy | 'uniform', 'distance' |
| `metric` | Distance metric | 'euclidean', 'manhattan', 'minkowski' |
| `p` | Minkowski power | 1 (manhattan), 2 (euclidean) |
| `algorithm` | Search algorithm | 'auto', 'ball_tree', 'kd_tree' |

### 3. Support Vector Machine
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `C` | Margin penalty | 0.1, 1, 10, 100 |
| `kernel` | Kernel function | 'rbf', 'linear', 'poly', 'sigmoid' |
| `gamma` | Kernel coefficient | 'scale', 'auto', 0.001–10 |
| `degree` | Poly kernel degree | 2, 3, 4 |
| `probability` | Enable predict_proba | True, False |

### 4. Decision Tree
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `criterion` | Split quality | 'gini', 'entropy' |
| `max_depth` | Tree depth limit | 2–20, None |
| `min_samples_split` | Min samples to split | 2, 5, 10 |
| `min_samples_leaf` | Min samples at leaf | 1, 2, 4 |
| `ccp_alpha` | Pruning parameter | 0.0–0.05 |

### 5. Random Forest
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `n_estimators` | Number of trees | 100, 200, 300, 500 |
| `max_features` | Features per split | 'sqrt', 'log2', None |
| `max_depth` | Each tree depth | 3, 5, 10, None |
| `bootstrap` | Use bootstrapping | True, False |
| `oob_score` | Out-of-bag eval | True, False |

### 6. AdaBoost
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `n_estimators` | Weak learners count | 50, 100, 200, 300 |
| `learning_rate` | Contribution shrinkage | 0.1, 0.5, 1.0, 1.5 |
| `algorithm` | Boosting algorithm | 'SAMME', 'SAMME.R' |
| `estimator` | Base learner | DecisionTreeClassifier(max_depth=1) |

### 7. Gradient Boosting
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `n_estimators` | Boosting stages | 100, 200, 300 |
| `learning_rate` | Step size | 0.01, 0.05, 0.1, 0.2 |
| `max_depth` | Tree depth | 2, 3, 4, 5 |
| `subsample` | Row sampling | 0.6, 0.8, 1.0 |
| `max_features` | Feature sampling | 'sqrt', 'log2' |

### 8. XGBoost
| Parameter | Description | Common Values |
|-----------|-------------|---------------|
| `n_estimators` | Number of trees | 100, 200, 300 |
| `learning_rate` (eta) | Step size | 0.01, 0.05, 0.1, 0.2 |
| `max_depth` | Tree depth | 3, 5, 7 |
| `subsample` | Row sampling | 0.6–1.0 |
| `colsample_bytree` | Column sampling | 0.6–1.0 |
| `gamma` | Min split loss | 0, 0.1, 0.3, 1 |
| `reg_alpha` | L1 regularization | 0, 0.1, 0.5, 1 |
| `reg_lambda` | L2 regularization | 1, 1.5, 2 |

### Hyperparameter Tuning: Grid Search CV



**Concept:** Tries **every possible combination** in the parameter grid.
- Advantages:
    - Guaranteed best combo in grid.
    - Easy to use.
- Disadvantages:
    - Slow for large grids
    - Exponential combinations

GridSearchCV(model, param_grid, cv=cv,scoring='accuracy', n_jobs=-1, verbose=0)

In [74]:
#  Helper Function 
def run_grid_search(name, model, param_grid, X_tr, y_tr, cv=5):
    gs = GridSearchCV(model, param_grid, cv=cv,
                      scoring='accuracy', n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr)
    print(f'\n{"="*52}')
    print(f'  {name}')
    print(f'  Best Params  : {gs.best_params_}')
    print(f'  Best CV Acc  : {gs.best_score_:.4f}')
    return gs.best_estimator_

print('Helper function defined')

Helper function defined


### 1. Logistic Regression — Grid Search

In [75]:
#  1. Logistic Regression — Grid Search 
lr_grid = {
    'C'        : [0.01, 0.1, 1, 10, 100],
    'penalty'  : ['l1', 'l2'],
    'solver'   : ['liblinear', 'saga'],
    'max_iter' : [500, 1000],
}
best_lr_gs = run_grid_search('Logistic Regression',
                              LogisticRegression(random_state=42),
                              lr_grid, X_train_scaled, y_train)


  Logistic Regression
  Best Params  : {'C': 0.1, 'max_iter': 500, 'penalty': 'l1', 'solver': 'saga'}
  Best CV Acc  : 0.9667


In [76]:
# 2. KNN — Grid Search 
knn_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan', 'minkowski'],
    'p'          : [1, 2],
}
best_knn_gs = run_grid_search('KNN',
                               KNeighborsClassifier(),
                               knn_grid, X_train_scaled, y_train)


  KNN
  Best Params  : {'metric': 'euclidean', 'n_neighbors': 5, 'p': 1, 'weights': 'uniform'}
  Best CV Acc  : 0.9667


In [77]:
# 3. SVM — Grid Search
svm_grid = {
    'C'     : [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma' : ['scale', 'auto'],
    'degree': [2, 3],
}
best_svm_gs = run_grid_search('SVM',
                               SVC(random_state=42),
                               svm_grid, X_train_scaled, y_train)


  SVM
  Best Params  : {'C': 0.1, 'degree': 2, 'gamma': 'scale', 'kernel': 'linear'}
  Best CV Acc  : 0.9750


In [48]:
# 4. Decision Tree — Grid Search 
dt_grid = {
    'criterion'        : ['gini', 'entropy'],
    'max_depth'        : [2, 3, 4, 5, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2', None],
}
best_dt_gs = run_grid_search('Decision Tree',
                              DecisionTreeClassifier(random_state=42),
                              dt_grid, X_train, y_train)


  Decision Tree
  Best Params  : {'criterion': 'gini', 'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}
  Best CV Acc  : 0.9667


In [49]:
#  5. Random Forest — Grid Search 
rf_grid = {
    'n_estimators'     : [100, 200, 300],
    'criterion'        : ['gini', 'entropy'],
    'max_depth'        : [3, 5, None],
    'max_features'     : ['sqrt', 'log2'],
    'min_samples_split': [2, 5],
    'bootstrap'        : [True, False],
}
best_rf_gs = run_grid_search('Random Forest',
                              RandomForestClassifier(random_state=42),
                              rf_grid, X_train, y_train)


  Random Forest
  Best Params  : {'bootstrap': True, 'criterion': 'gini', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 100}
  Best CV Acc  : 0.9583


In [50]:
# 6. AdaBoost — Grid Search 
ada_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.5, 1.0, 1.5, 2.0],
    'algorithm'   : ['SAMME', 'SAMME.R'],
}
best_ada_gs = run_grid_search('AdaBoost',
                               AdaBoostClassifier(random_state=42),
                               ada_grid, X_train, y_train)


  AdaBoost
  Best Params  : {'algorithm': 'SAMME', 'learning_rate': 0.5, 'n_estimators': 100}
  Best CV Acc  : 0.9583


In [ ]:
#  7. Gradient Boosting — Grid Search ( Not Good)
gb_grid = {
    'n_estimators' : [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth'    : [2, 3, 4],
    'subsample'    : [0.7, 0.8, 1.0],
    'max_features' : ['sqrt', 'log2'],
}
best_gb_gs = run_grid_search('Gradient Boosting',
                              GradientBoostingClassifier(random_state=42),
                              gb_grid, X_train, y_train)

In [ ]:
# #  8. XGBoost — Grid Search (Not Good to used)
xgb_grid = {
    'n_estimators'    : [100, 200],
    'learning_rate'   : [0.05, 0.1, 0.2],
    'max_depth'       : [3, 5, 7],
    'subsample'       : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'gamma'           : [0, 0.1, 0.3],
    'reg_alpha'       : [0, 0.1],
}
best_xgb_gs = run_grid_search('XGBoost',
                               XGBClassifier(random_state=42,
                                             use_label_encoder=False,
                                             eval_metric='mlogloss'),
                               xgb_grid, X_train, y_train)

In [78]:
#  Final Test Accuracy — Grid Search Tuned Models 
print('\n' + '='*52)
print('  FINAL TEST ACCURACY — GRID SEARCH TUNED MODELS')
print('='*52)

gs_best = {
    'Logistic Regression': (best_lr_gs,  X_test_sc),
    'KNN'                : (best_knn_gs, X_test_sc),
    'SVM'                : (best_svm_gs, X_test_sc),
    'Decision Tree'      : (best_dt_gs,  X_test),
    'Random Forest'      : (best_rf_gs,  X_test),
    'AdaBoost'           : (best_ada_gs, X_test),
    # 'Gradient Boosting'  : (best_gb_gs,  X_test),
    # 'XGBoost'            : (best_xgb_gs, X_test),
}

gs_results = {}
for name, (model, Xte) in gs_best.items():
    acc = accuracy_score(y_test, model.predict(Xte))
    gs_results[name] = acc
    print(f'  {name:<24}  →  Test Acc: {acc:.4f}')

print(f'\n Best: {max(gs_results, key=gs_results.get)} ({max(gs_results.values()):.4f})')


  FINAL TEST ACCURACY — GRID SEARCH TUNED MODELS
  Logistic Regression       →  Test Acc: 0.9000
  KNN                       →  Test Acc: 0.9333
  SVM                       →  Test Acc: 0.9333
  Decision Tree             →  Test Acc: 0.9333
  Random Forest             →  Test Acc: 0.9667
  AdaBoost                  →  Test Acc: 0.9667

 Best: Random Forest (0.9667)


### Hyperparameter Tuning: Randomized Search CV

**Concept:** Randomly samples `n_iter` combinations from the parameter distributions.  

- Advantages:

- Much faster than Grid Search.
- Works with continuous distributions.
- Scales to large param spaces 

- Disadvantages:

- Not guaranteed to find absolute best



RandomizedSearchCV(model, param_dist, n_iter=n_iter,
                            cv=cv, scoring='accuracy',
                            n_jobs=-1, random_state=42, verbose=0)

| Parameter      | Purpose                     |
| -------------- | --------------------------- |
| `model`        | What to train               |
| `param_dist`   | What to tune                |
| `n_iter`       | How many trials             |
| `cv`           | How to validate             |
| `scoring`      | How to judge performance    |
| `n_jobs`       | Speed (parallel processing) |
| `random_state` | Reproducibility             |
| `verbose`      | Debug visibility            |


In [79]:
#  Helper Function 
def run_random_search(name, model, param_dist, X_tr, y_tr, n_iter=50, cv=5):
    rs = RandomizedSearchCV(model, param_dist, n_iter=n_iter,
                            cv=cv, scoring='accuracy',
                            n_jobs=-1, random_state=42, verbose=0)
    rs.fit(X_tr, y_tr)
    print(f'\n{"="*52}')
    print(f'  {name}  (n_iter={n_iter})')
    print(f'  Best Params  : {rs.best_params_}')
    print(f'  Best CV Acc  : {rs.best_score_:.4f}')
    return rs.best_estimator_

print('Helper function defined')

Helper function defined


###  1. Logistic Regression — Random Search 


In [ ]:
#  1. Logistic Regression — Random Search 
lr_dist = {
    'C'       : loguniform(1e-3, 1e3),
    'penalty' : ['l1', 'l2'],
    'solver'  : ['liblinear', 'saga'],
    'max_iter': randint(100, 2000),
}
best_lr_rs = run_random_search('Logistic Regression',
                                LogisticRegression(random_state=42),
                                lr_dist, X_train_scaled, y_train, n_iter=30)


  Logistic Regression  (n_iter=30)
  Best Params  : {'C': np.float64(47.6591180868084), 'max_iter': 1144, 'penalty': 'l1', 'solver': 'saga'}
  Best CV Acc  : 0.9667


###  2. KNN — Random Search 


In [ ]:
#  2. KNN — Random Search 
knn_dist = {
    'n_neighbors': randint(2, 25),
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan', 'chebyshev'],
    'p'          : [1, 2, 3],
    'leaf_size'  : randint(10, 60),
}
best_knn_rs = run_random_search('KNN',
                                 KNeighborsClassifier(),
                                 knn_dist, X_train_scaled, y_train, n_iter=30)


  KNN  (n_iter=30)
  Best Params  : {'leaf_size': 49, 'metric': 'euclidean', 'n_neighbors': 17, 'p': 1, 'weights': 'distance'}
  Best CV Acc  : 0.9750


###  3. SVM — Random Search 


In [82]:
#  3. SVM — Random Search 
svm_dist = {
    'C'     : loguniform(0.01, 1000),
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma' : loguniform(1e-4, 10),
    'degree': randint(2, 5),
}
best_svm_rs = run_random_search('SVM',
                                 SVC(random_state=42),
                                 svm_dist, X_train_scaled, y_train, n_iter=40)


  SVM  (n_iter=40)
  Best Params  : {'C': np.float64(75.10418138777538), 'degree': 3, 'gamma': np.float64(0.009456951897345888), 'kernel': 'sigmoid'}
  Best CV Acc  : 0.9750


###  4. Decision Tree — Random Search 


In [83]:
#  4. Decision Tree — Random Search 
dt_dist = {
    'criterion'        : ['gini', 'entropy'],
    'max_depth'        : [None] + list(range(2, 20)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf' : randint(1, 10),
    'max_features'     : ['sqrt', 'log2', None],
    'ccp_alpha'        : uniform(0, 0.05),
}
best_dt_rs = run_random_search('Decision Tree',
                                DecisionTreeClassifier(random_state=42),
                                dt_dist, X_train, y_train, n_iter=50)


  Decision Tree  (n_iter=50)
  Best Params  : {'ccp_alpha': np.float64(0.007143340896097039), 'criterion': 'gini', 'max_depth': 2, 'max_features': 'log2', 'min_samples_leaf': 6, 'min_samples_split': 3}
  Best CV Acc  : 0.9667


###  5. Random Forest — Random Search 


In [84]:
#  5. Random Forest — Random Search 
rf_dist = {
    'n_estimators'     : randint(50, 500),
    'criterion'        : ['gini', 'entropy'],
    'max_depth'        : [None] + list(range(2, 20)),
    'max_features'     : ['sqrt', 'log2', None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf' : randint(1, 10),
    'bootstrap'        : [True, False],
}
best_rf_rs = run_random_search('Random Forest',
                                RandomForestClassifier(random_state=42),
                                rf_dist, X_train, y_train, n_iter=50)


  Random Forest  (n_iter=50)
  Best Params  : {'bootstrap': False, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 3, 'min_samples_split': 8, 'n_estimators': 70}
  Best CV Acc  : 0.9583


###  6. AdaBoost — Random Search 


In [85]:
#  6. AdaBoost — Random Search 
ada_dist = {
    'n_estimators': randint(50, 500),
    'learning_rate': loguniform(0.01, 2.0),
    'algorithm'   : ['SAMME', 'SAMME.R'],
}
best_ada_rs = run_random_search('AdaBoost',
                                 AdaBoostClassifier(random_state=42),
                                 ada_dist, X_train, y_train, n_iter=30)


  AdaBoost  (n_iter=30)
  Best Params  : {'algorithm': 'SAMME', 'learning_rate': np.float64(0.2500224047333955), 'n_estimators': 70}
  Best CV Acc  : 0.9667


###  7. Gradient Boosting — Random Search 


In [86]:
#  7. Gradient Boosting — Random Search 
gb_dist = {
    'n_estimators'     : randint(50, 500),
    'learning_rate'    : loguniform(0.01, 0.5),
    'max_depth'        : randint(2, 8),
    'subsample'        : uniform(0.5, 0.5),
    'min_samples_split': randint(2, 20),
    'max_features'     : ['sqrt', 'log2', None],
}
best_gb_rs = run_random_search('Gradient Boosting',
                                GradientBoostingClassifier(random_state=42),
                                gb_dist, X_train, y_train, n_iter=50)


  Gradient Boosting  (n_iter=50)
  Best Params  : {'learning_rate': np.float64(0.05954553793888989), 'max_depth': 7, 'max_features': None, 'min_samples_split': 13, 'n_estimators': 104, 'subsample': np.float64(0.9916154429033941)}
  Best CV Acc  : 0.9667


###  8. XGBoost — Random Search 


In [87]:
#  8. XGBoost — Random Search 
xgb_dist = {
    'n_estimators'    : randint(50, 500),
    'learning_rate'   : loguniform(0.005, 0.5),
    'max_depth'       : randint(2, 12),
    'subsample'       : uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.4, 0.6),
    'gamma'           : uniform(0, 1),
    'reg_alpha'       : loguniform(1e-3, 10),
    'reg_lambda'      : loguniform(1e-3, 10),
    'min_child_weight': randint(1, 10),
}
best_xgb_rs = run_random_search('XGBoost',
                                 XGBClassifier(random_state=42,
                                               use_label_encoder=False,
                                               eval_metric='mlogloss'),
                                 xgb_dist, X_train, y_train, n_iter=60)


  XGBoost  (n_iter=60)
  Best Params  : {'colsample_bytree': np.float64(0.5825453457757226), 'gamma': np.float64(0.5247564316322378), 'learning_rate': np.float64(0.03654769917956455), 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 413, 'reg_alpha': np.float64(0.11400863701127326), 'reg_lambda': np.float64(0.23423849847112907), 'subsample': np.float64(0.5232252063599989)}
  Best CV Acc  : 0.9667
